# Corrective RAG (CRAG)

> Grade the retriever's output, then correct course - augment with web search, or rewrite the query and retry.

![CRAG](../assets/crag.png)

Corrective-RAG (CRAG) is a strategy for RAG that incorporates self-reflection / self-grading on retrieved documents.

The paper follows this general flow:

- If at least one document exceeds the threshold for relevance, then it proceeds to generation.
- If all documents fall below the relevance threshold or if the grader is unsure, then it uses web search to supplement retrieval.
- Before generation, it performs knowledge refinement of the search or retrieved documents - partition into knowledge strips, grade each strip, filter out irrelevant ones.

What this notebook implements:

- Title extraction → MyAnimeList (Jikan API) lookup → index → retrieve → grade → route (generate / web-search / rewrite).
- Web-search supplement via Tavily whenever the grader says AMBIGUOUS or INCORRECT.
- Knowledge refinement is **skipped** it can be added back later as a node if desired.
- Domain: anime questions grounded in MyAnimeList data.

In [2]:
import os
import re
import json
import time
import operator
import uuid
from pathlib import Path
from typing import Optional, Literal, Annotated, TypedDict

import requests
import chromadb
from dotenv import load_dotenv
from pydantic import BaseModel
from rapidfuzz import fuzz

from langchain_core.documents import Document
from langchain_core.runnables import RunnableConfig
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START, END

load_dotenv("../../.env")

if not os.getenv("TAVILY_API_KEY"):
    print("WARNING: TAVILY_API_KEY missing")

## Models

| Role | Model ID | Params | Notes |
|---|---|---|---|
| Fast | `openai/gpt-oss-20b` | 20B | Extract, grade, rewrite |
| Smart | `openai/gpt-oss-120b` | 120B | Generate only |
| Embeddings | `BAAI/bge-small-en-v1.5` | 33M | CPU, HuggingFaceEmbeddings |

### Two gpt-oss quirks that will bite you if you don't plan for them

1. **It is a reasoning model.** Reasoning text arrives in `additional_kwargs["reasoning_content"]` (langchain-groq maps Groq's `reasoning` field there); `response_metadata` only carries `reasoning_effort` and `token_usage ... reasoning_tokens`. Nodes read `response.content` for the answer and **ignore** reasoning but test cells print reasoning once so you can see the model "think." It is also slower per call than Llama 8B was.


2. **`reasoning_effort` param.** Groq accepts `reasoning_effort="low"|"medium"|"high"` in `ChatGroq(...)`. Plan per node:

| Node | effort | Why |
|---|---|---|
| `extract_title`, `rewrite_query` | low | deterministic, cheap |
| `grade_documents` | medium | needs actual judgment |
| `generate` | low | you want verbose *output*, not verbose thinking - the 120B's raw writing is already good |

In [5]:
# Fast lane: extraction + rewrite (deterministic, cheap)
llm_fast = ChatGroq(model="openai/gpt-oss-20b", temperature=0, reasoning_effort="low")

# Grader: same weights, medium - this node has to actually judge relevance
llm_grade = ChatGroq(model="openai/gpt-oss-20b", temperature=0, reasoning_effort="medium")

# Smart lane: generation only
llm_smart = ChatGroq(model="openai/gpt-oss-120b", temperature=0, reasoning_effort="low")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## State schema

Defined in the next cell. Most fields use default **replace** semantics; two use `Annotated[list[Document], operator.add]`.

### Why these two fields use reducers

- **`mal_docs`** : the node runs once per invocation, but if the graph ever loops back through `jikan_lookup` (it won't in this design, but defensive), you don't want to silently drop the first result. Also makes the trace clearer: every append is a visible state delta.

- **`web_docs`** : this one genuinely matters. On the AMBIGUOUS/INCORRECT path, `web_search` may fire twice (initial + after rewrite). Without the reducer, the second call replaces the first result set and your grader only sees half the evidence. With the reducer, both result sets accumulate.

### Why everything else is NOT a reducer

- Default replace is correct when a node computes the whole new value for a field, not a delta.

- **`retrieved_docs` and `doc_grades` are recomputed fresh on every `index_and_retrieve` / `grade_documents` pass.** Made additive, you would grade stale docs from a previous retry  a subtle bug that produces confusing traces.

- Don't reach for reducers reflexively; only use them when a node appends.

In [7]:
class CRAGState(TypedDict):
    # ---- input ----
    user_query: str

    # ---- extraction ----
    anime_title: str
    anime_year: Optional[int]

    # ---- mal resolution ----
    mal_id: Optional[int]
    mal_docs: Annotated[list[Document], operator.add]

    # ---- corrective path ----
    web_docs: Annotated[list[Document], operator.add]
    web_used: bool
    rewritten_query: Optional[str]
    retry_count: int

    # ---- retrieval + grading ----
    retrieved_docs: list[Document]
    doc_grades: list[dict]
    retrieval_quality: Literal["CORRECT", "AMBIGUOUS", "INCORRECT", "UNKNOWN"]

    # ---- output ----
    final_answer: str

## Why every node takes a config

LangGraph forwards a config dict into each node as the second positional argument, so nodes are `def node(state, config)` throughout  you just have to accept it.

Every node builds a config carrying a stable `run_name` and `tags` (and passes it through to `.invoke()`), which gives you, in LangSmith:

- one identifiable span per stage (`extract_title`, `grade_documents`, ...),
- filterable tags (`stage:extract`, `llm:gpt-oss-20b`, ...),
- `user_query` attached as metadata on every span.

Non-LLM nodes (`jikan_lookup`, `index_and_retrieve`) accept `config` too - same rule: every node invocation is traced, not just the LLM calls.

In [3]:
def trace_config(config: RunnableConfig, run_name: str, tags: list[str], user_query: str = "", **extra) -> dict:
    """Merge the graph-provided config with a stable run_name + tags for LangSmith."""
    return {
        **config,
        "run_name": run_name,
        "tags": tags,
        "metadata": {**config.get("metadata", {}), "user_query": user_query, **extra},
    }

### `extract_title`

| | |
|---|---|
| Reads | `user_query` |
| Writes | `anime_title`, `anime_year` |
| Model | gpt-oss-20b, `reasoning_effort="low"`, structured output |
| LangSmith tag | `{"stage": "extract", "llm": "gpt-oss-20b"}` |

Schema (Pydantic):

```python
class TitleExtraction(BaseModel):
    title: str          # empty string if no title found
    year: Optional[int] = None
    confidence: float   # 0.0–1.0, self-reported
```

**Failure mode:** if `confidence < 0.4` or `title == ""`, still write a best guess `anime_title = user_query[:80]` as a naive fallback so `jikan_lookup` has something to search. It will almost certainly miss, which triggers the corrective path. That is correct CRAG behavior: don't bail early, let the grader decide.

In [ ]:
class TitleExtraction(BaseModel):
    title: str          # empty string if no title found
    year: Optional[int] = None
    confidence: float   # 0.0-1.0, self-reported

structured_extract = llm_fast.with_structured_output(TitleExtraction)

EXTRACT_PROMPT = """You extract the anime title and release year from a user's question.

Question: {query}

Rules:
- title: the anime being asked about; "" if the question names no anime.
- year: the 4-digit release year if the question states one, else null.
- confidence: your self-reported certainty, 0.0-1.0.
Return only the structured object."""

def extract_title(state: CRAGState, config: RunnableConfig) -> dict:
    
    cfg = trace_config(
        config, "extract_title",
        ["crag", "stage:extract", "llm:gpt-oss-20b"],
        user_query=state["user_query"],
    )

    prompt = EXTRACT_PROMPT.format(query=state["user_query"])
    result = structured_extract.invoke([("user", prompt)], config=cfg)

    # nodes read the parsed output 
    if result.title and result.confidence >= 0.4:
        title = result.title
    else:
        # naive fallback: give jikan something to search; a miss here is
        # correct CRAG behavior - the grader decides, not this node
        title = state["user_query"][:80]
    return {"anime_title": title, "anime_year": result.year}

### Test: watch it think

One live call that prints the reasoning once (`additional_kwargs["reasoning_content"]`) - proof the model actually reasons - then the structured extraction through the real node.

In [ ]:
# testing the model flow with a simple query
raw = llm_fast.invoke(
    [("user", "What is Steins;Gate about? Reply with only the anime title.")],
    config=trace_config({}, "test:extract_thinking", ["crag", "test"], user_query="What is Steins;Gate about?"),
)

meta = raw.response_metadata
reasoning = raw.additional_kwargs.get("reasoning_content") or meta.get("reasoning") or ""
    
print("REASONING >>>", reasoning)
print("CONTENT  >>>", raw.content)


REASONING >>> User wants only the anime title. So reply "Steins;Gate".
CONTENT  >>> Steins;Gate
{'anime_title': 'Steins;Gate', 'anime_year': None}


In [10]:
# structured extraction through the actual node
out = extract_title({"user_query": "What is the best anime all time"}, {})
print(out)   

{'anime_title': 'What is the best anime all time', 'anime_year': None}


### `jikan_lookup`

| Field | Value |
|---|---|
| Reads | `anime_title`, `anime_year` |
| Writes | `mal_id`, `mal_docs` |
| Model | none / pure HTTP |
| LangSmith tag | `{"stage": "mal", "source": "jikan"}` |

Calls:

1. `GET /anime?q={title}&limit=5&sfw=true` → pick best match
2. if match: `GET /anime/{id}/full`
3. if match: `GET /anime/{id}/characters`

**Match rule:** `rapidfuzz.fuzz.WRatio(title_lower, candidate_title_lower) >= 80`. If `anime_year` is set, prefer candidates whose year matches first, then fall back to WRatio. Never call the LLM here  deterministic beats probabilistic for structured APIs, and it keeps the LangSmith trace clean (a "lookup" span should not contain an LLM span).

**Failure mode:** if no candidate passes 80, write `mal_id=None`, `mal_docs=[]`. Do not raise the downstream conditional edge detects `mal_id is None`.

**Rate limit:** `time.sleep(0.4)` before every Jikan call (60/min cap). Two calls per successful lookup = 0.8s minimum per query. Fine.

#### Documents produced (metadata contract - do not deviate)

| Section | Content | Source endpoint |
|---|---|---|
| `synopsis` | `data["synopsis"]` (HTML stripped) | `/full` |
| `metadata` | title, title_english, title_japanese, type, episodes, status, aired, duration, rating, score, rank, popularity, members, favorites, studios, producers, genres, themes, demographics | `/full` |
| `staff` | serialized staff \ name + role | `/full` |
| `characters` | serialized characters \ name + role + favorites | `/characters` |

Four documents per successful lookup, one per section. Each carries the full metadata block:

```python
{"source": "mal", "mal_id": int, "section": "synopsis"|"metadata"|"staff"|"characters",
 "title": str,   # canonical title from MAL
 "url": str}     # https://myanimelist.net/anime/{id}
```

Staff and characters truncate to the top 15 entries each full lists blow past the ~4k-token budget (see the LangSmith checklist at the bottom).

In [22]:
JIKAN = "https://api.jikan.moe/v4"
MATCH_THRESHOLD = 80
SECTION_LIMIT = 15   # top-N only

def _jikan_get(path: str, **params) -> dict:
    time.sleep(0.4)
    resp = requests.get(f"{JIKAN}{path}", params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()

def _strip_html(text: Optional[str]) -> str:
    return re.sub(r"<[^>]+>", "", text or "").strip()

def _match_score(title: str, candidate: str) -> float:
    return fuzz.WRatio(title.lower(), candidate.lower())

def _staff_rows(items: Optional[list]) -> list:
    rows = []
    for s in items or []:
        positions = s.get("positions") or ([s.get("position")] if s.get("position") else [])
        rows.append({"name": s.get("name"), "role": ", ".join(p for p in positions if p)})
    return rows[:SECTION_LIMIT]

def _character_rows(items: Optional[list]) -> list:
    rows = []
    for c in items or []:
        ch = c.get("character") or {}
        favorites = c.get("favorites")
        if favorites is None:
            favorites = ch.get("favorite")
        rows.append({"name": ch.get("name"), "role": c.get("role"), "favorites": favorites})
    return rows[:SECTION_LIMIT]


In [17]:

def jikan_lookup(state: CRAGState, config: RunnableConfig) -> dict:
    cfg = trace_config(
        config, "jikan_lookup",
        ["crag", "stage:mal", "source:jikan"],
        user_query=state["user_query"],
    )  
    title, year = state["anime_title"], state.get("anime_year")

    try:
        search = _jikan_get("/anime", q=title, limit=5, sfw=True)
    except Exception as exc:
        print(f"jikan search failed: {exc}")
        return {"mal_id": None, "mal_docs": []}   # never raise Edge A handles None

    candidates = search.get("data") or []
    scored = sorted(candidates, key=lambda c: _match_score(title, c.get("title", "")), reverse=True)

    # year match first (when we have a year), then plain WRatio both still need >= 80
    matched = None
    if year is not None:
        for cand in scored:
            if cand.get("year") == year and _match_score(title, cand.get("title", "")) >= MATCH_THRESHOLD:
                matched = cand
                break

    if matched is None:
        for cand in scored:
            if _match_score(title, cand.get("title", "")) >= MATCH_THRESHOLD:
                matched = cand
                break
            
    if matched is None:
        return {"mal_id": None, "mal_docs": []}

    mal_id = matched["id"]
    try:
        full = _jikan_get(f"/anime/{mal_id}/full")["data"]
        chars = _jikan_get(f"/anime/{mal_id}/characters")["data"]
    except Exception as exc:
        print(f"jikan detail fetch failed: {exc}")
        return {"mal_id": None, "mal_docs": []}   # partial data is worse than none

    base = {
        "source": "mal",
        "mal_id": mal_id,
        "title": full.get("title") or matched.get("title"),
        "url": f"https://myanimelist.net/anime/{mal_id}",
    }
    meta_block = {
        k: full.get(k)
        for k in ["title", "title_english", "title_japanese", "type", "episodes", "status",
                  "aired", "duration", "rating", "score", "rank", "popularity", "members",
                  "favorites", "studios", "producers", "genres", "themes", "demographics"]
    }
    if isinstance(meta_block.get("aired"), dict):
        meta_block["aired"] = meta_block["aired"].get("string")

    docs = [
        Document(page_content=_strip_html(full.get("synopsis")),
                 metadata={**base, "section": "synopsis"}),
        Document(page_content=json.dumps(meta_block, ensure_ascii=False, default=str),
                 metadata={**base, "section": "metadata"}),
        Document(page_content=json.dumps(_staff_rows(full.get("staff")), ensure_ascii=False),
                 metadata={**base, "section": "staff"}),
        Document(page_content=json.dumps(_character_rows(chars), ensure_ascii=False), 
                 metadata={**base, "section": "characters"}),
    ]
    return {"mal_id": mal_id, "mal_docs": docs}

### Test: exact-title match

`"Steins;Gate"` vs the Jikan result must score **100** on WRatio - punctuation included - and yield exactly the four contracted sections.

In [23]:
out = jikan_lookup(
    {"user_query": "What is Steins;Gate about?", "anime_title": "Steins;Gate", "anime_year": None},
    {},
)
assert out["mal_id"] is not None, "Steins;Gate must match on Jikan (if None: Jikan unreachable or title-match bug)"
assert len(out["mal_docs"]) == 4, "exactly four sections per the contract"
assert {d.metadata["section"] for d in out["mal_docs"]} == {"synopsis", "metadata", "staff", "characters"}

print("mal_id:", out["mal_id"])
for d in out["mal_docs"]:
    print(f"  [{d.metadata['section']:>10}] {len(d.page_content):>5} chars | {d.metadata['url']}")
print("WRatio:", fuzz.WRatio("steins;gate", (out["mal_docs"][0].metadata["title"] or "").lower()))

jikan search failed: HTTPSConnectionPool(host='api.jikan.moe', port=443): Max retries exceeded with url: /v4/anime?q=Steins%3BGate&limit=5&sfw=True (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.jikan.moe', port=443) at 0x1d632be7fb0>, 'Connection to api.jikan.moe timed out. (connect timeout=60)'))


AssertionError: Steins;Gate must match on Jikan (if None: Jikan unreachable or title-match bug)

### `web_search`

| | |
|---|---|
| Reads | `user_query`, `anime_title`, `anime_year`, `rewritten_query` |
| Writes | `web_docs` (**reducer → appends**), `web_used=True` |
| Model | none |
| LangSmith tag | `{"stage": "web", "provider": "tavily"}` |

Query construction (exact):

```python
q = state.get("rewritten_query") or f'{state["anime_title"]} anime {state.get("anime_year") or ""} synopsis characters'.strip()
```

- Tavily config: `max_results=4`, `search_depth="advanced"`, `include_raw_content=True`.
- One `Document` per Tavily result; `page_content` = `result["content"]` (+ `raw_content` if present, truncated to 2000 chars).
- Metadata: `{"source": "web", "url", "title", "score"}` - plus `section="web"` (see schema below).

**Failure mode:** Tavily returns an empty list → append nothing but still set `web_used=True`. The grader will see zero relevant docs and route to INCORRECT, which forces generate with the low-confidence warning (because `retry_count >= 1` by then).

In [ ]:
tavily = TavilySearch(max_results=4, search_depth="advanced", include_raw_content=True)

def web_search(state: CRAGState, config: RunnableConfig) -> dict:
    cfg = trace_config(
        config, "web_search",
        ["crag", "stage:web", "provider:tavily"],
        user_query=state["user_query"],
    )
    # rewritten_query wins - that is the whole point of the rewrite step
    q = state.get("rewritten_query") or (
        f'{state["anime_title"]} anime {state.get("anime_year") or ""} synopsis characters'.strip()
    )
    try:
        results = (tavily.invoke({"query": q}, config=cfg) or {}).get("results") or []
    except Exception as exc:
        print(f"web_search failed: {exc}")
        results = []          # empty + web_used=True → grader routes to INCORRECT
    docs = []
    for r in results:
        content = r.get("content") or ""
        raw = (r.get("raw_content") or "")[:2000]
        if raw:
            content = f"{content}\n\n{raw}"
        docs.append(Document(
            page_content=content,
            metadata={
                "source": "web",
                "url": r.get("url"),
                "title": r.get("title"),
                "score": r.get("score"),
                "section": "web",
            },
        ))
    return {"web_docs": docs, "web_used": True}   # web_docs is a reducer - appends, never replaces

## Document metadata schema - locked

Every `Document` in either `mal_docs` or `web_docs` carries:

| Key | Type | MAL | Web | Purpose |
|---|---|---|---|---|
| `source` | `"mal"` \| `"web"` | ✓ | ✓ | citation rendering, grading transparency |
| `section` | enum (below) | ✓ | ✓ | section routing |
| `mal_id` | int | ✓ | ✗ | dedup, re-fetch |
| `title` | str | ✓ | ✓ | display |
| `url` | str | ✓ (MAL page) | ✓ (result URL) | citation link |
| `score` | float | ✗ | ✓ | Tavily relevance, logged only |

`section` enum: `"synopsis"`, `"metadata"`, `"staff"`, `"characters"` (MAL only), `"web"` (web only - a placeholder so downstream code never has to check for a missing key).

**Never store `doc_grades` in Chroma.** Grades live in state only. Chroma is a retrieval cache, not a grading cache - mixing the two is the #1 way learners turn CRAG into a mess.

### `index_and_retrieve`

| | |
|---|---|
| Reads | `user_query`, `mal_docs`, `web_docs` |
| Writes | `retrieved_docs` (**replace** - recomputed every pass) |
| Model | none (embeddings only) |
| LangSmith tag | `{"stage": "retrieve"}` |

Behavior:

- `chromadb.Client().create_collection(name=f"crag_{uuid4().hex[:8]}")` - fresh collection per invocation. No collision risk, no cleanup needed, matches the `persistence=False` decision.
- Embed `mal_docs + web_docs` with `HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")`.
- Query with `user_query` for `k=min(6, total_docs)`.
- Map hit ids back to the original `Document` objects so metadata survives intact.

**Why index-then-retrieve when we already hold the docs?** Because this is CRAG, and CRAG's whole thesis is that grading happens *after* a retriever has done its job. Bypassing retrieval collapses the architecture into "fetch-and-stuff," which is a different (and worse) system. This is the most common shortcut learners take - the graph keeps the retriever in the loop on purpose.

**Failure mode:** zero docs (both sources empty) → `retrieved_docs=[]` → grader handles it.

In [ ]:
def index_and_retrieve(state: CRAGState, config: RunnableConfig) -> dict:
    cfg = trace_config(
        config, "index_and_retrieve",
        ["crag", "stage:retrieve"],
        user_query=state["user_query"],
    )  # no child runnable receives cfg - the node itself is the traced span
    docs = [d for d in [*state.get("mal_docs", []), *state.get("web_docs", [])] if d.page_content.strip()]
    if not docs:
        return {"retrieved_docs": []}

    collection = chromadb.Client().create_collection(name=f"crag_{uuid.uuid4().hex[:8]}")  # fresh per invocation
    texts = [d.page_content for d in docs]
    collection.add(
        ids=[str(i) for i in range(len(docs))],
        documents=texts,
        embeddings=embeddings.embed_documents(texts),
    )
    k = min(6, len(docs))
    hits = collection.query(
        query_embeddings=embeddings.embed_query(state["user_query"]),
        n_results=k,
    )
    retrieved = [docs[int(i)] for i in hits["ids"][0]]   # original objects → metadata preserved
    return {"retrieved_docs": retrieved}

### `grade_documents`

| | |
|---|---|
| Reads | `user_query`, `retrieved_docs` |
| Writes | `doc_grades`, `retrieval_quality` (both **replace** - recomputed per pass) |
| Model | gpt-oss-20b, `reasoning_effort="medium"`, structured output |
| LangSmith tag | `{"stage": "grade", "llm": "gpt-oss-20b"}` |

One call grades all documents (`GradeBatch`):

```python
class DocGrade(BaseModel):
    index: int
    relevant: bool
    reason: str   # one sentence

class GradeBatch(BaseModel):
    grades: list[DocGrade]
```

Classification rule - **computed in Python, not by the LLM**:

```python
relevant = [g for g in doc_grades if g["relevant"]]
n, r = len(doc_grades), len(relevant)
if n == 0 or r == 0:   quality = "INCORRECT"
elif r == n:            quality = "CORRECT"
else:                   quality = "AMBIGUOUS"
```

**Why this matters:** the LLM grades each document independently; the policy decision (CORRECT/AMBIGUOUS/INCORRECT) is deterministic. If you let the LLM also pick the bucket, you get inconsistent behavior across runs - the CRAG paper is explicit about this separation.

**Failure mode:** if the LLM returns malformed JSON → repair with a regex extract of the first `[...]` array, else mark all docs `relevant=False` and set quality INCORRECT. Never raise.

In [ ]:
class DocGrade(BaseModel):
    index: int
    relevant: bool
    reason: str   # one sentence

class GradeBatch(BaseModel):
    grades: list[DocGrade]

structured_grade = llm_grade.with_structured_output(GradeBatch)

GRADE_PROMPT = """You are grading retrieved documents for a RAG system.

User query: {query}

Documents:
{doc_lines}

For each document, decide if it is relevant to answering the query.
Return one grade per document, keyed by index."""

def _repair_grades(raw: str) -> list[dict]:
    """Regex-extract the first [...] array from a malformed response."""
    match = re.search(r"\[.*\]", raw, re.DOTALL)
    if not match:
        return []
    try:
        parsed = json.loads(match.group(0))
        return [{"index": int(g["index"]), "relevant": bool(g["relevant"]), "reason": str(g.get("reason", ""))}
                for g in parsed if isinstance(g, dict) and "index" in g and "relevant" in g]
    except Exception:
        return []

def grade_documents(state: CRAGState, config: RunnableConfig) -> dict:
    cfg = trace_config(
        config, "grade_documents",
        ["crag", "stage:grade", "llm:gpt-oss-20b"],
        user_query=state["user_query"],
    )
    docs = state.get("retrieved_docs", [])
    n = len(docs)
    if n == 0:
        return {"doc_grades": [], "retrieval_quality": "INCORRECT"}

    lines = [
        f"[{i}] (source={d.metadata.get('source')}, section={d.metadata.get('section')}) {d.page_content[:500]}"
        for i, d in enumerate(docs)
    ]
    prompt = GRADE_PROMPT.format(query=state["user_query"], doc_lines="\n".join(lines))

    try:
        batch = structured_grade.invoke([("user", prompt)], config=cfg)
        grades = [{"index": g.index, "relevant": g.relevant, "reason": g.reason} for g in batch.grades]
    except Exception as exc:
        # malformed output → regex repair of the first [...] → else fall through to all-False
        print(f"structured grade failed ({exc}); attempting regex repair")
        raw = llm_grade.invoke([("user", prompt)], config=cfg)
        raw_text = raw.content if isinstance(raw.content, str) else str(raw.content)
        grades = _repair_grades(raw_text)

    by_index = {g["index"]: g for g in grades if 0 <= g["index"] < n}
    doc_grades = [
        by_index.get(i, {"index": i, "relevant": False, "reason": "Grader returned no grade for this document."})
        for i in range(n)
    ]

    # the policy stays deterministic - the LLM grades, it never classifies
    r = sum(1 for g in doc_grades if g["relevant"])
    if r == 0:
        quality = "INCORRECT"
    elif r == n:
        quality = "CORRECT"
    else:
        quality = "AMBIGUOUS"
    return {"doc_grades": doc_grades, "retrieval_quality": quality}

### Test: hand-crafted batch

Three relevant documents + one irrelevant document for the Cowboy Bebop question → the batch schema must return **4 grades** and the deterministic policy must compute **AMBIGUOUS**.

In [ ]:
test_docs = [
    Document(
        page_content="Cowboy Bebop is a 1998 anime television series animated by Sunrise and directed by Shinichiro Watanabe.",
        metadata={"source": "mal", "section": "metadata", "mal_id": 1, "title": "Cowboy Bebop",
                  "url": "https://myanimelist.net/anime/1"},
    ),
    Document(
        page_content='[{"name": "Yoko Kanno", "role": "Music"}, {"name": "Keiko Nobumoto", "role": "Series Composition"}]',
        metadata={"source": "mal", "section": "staff", "mal_id": 1, "title": "Cowboy Bebop",
                  "url": "https://myanimelist.net/anime/1"},
    ),
    Document(
        page_content="Bounty hunter Spike Spiegel chases criminals across the solar system with his crew aboard the spaceship Bebop.",
        metadata={"source": "mal", "section": "synopsis", "mal_id": 1, "title": "Cowboy Bebop",
                  "url": "https://myanimelist.net/anime/1"},
    ),
    Document(
        page_content="Steins;Gate follows Rintaro Okabe, who discovers he can send messages to the past through a microwave.",
        metadata={"source": "web", "section": "web", "title": "Steins;Gate wiki",
                  "url": "https://example.com/steins-gate", "score": 0.91},
    ),
]
out = grade_documents(
    {"user_query": "What year did Cowboy Bebop air and who composed its music?", "retrieved_docs": test_docs},
    {},
)
for g in out["doc_grades"]:
    print(g["index"], g["relevant"], "|", g["reason"])
print("quality:", out["retrieval_quality"])

assert len(out["doc_grades"]) == 4, "batch must return one grade per document"
assert out["retrieval_quality"] == "AMBIGUOUS", "3 relevant + 1 irrelevant must compute to AMBIGUOUS"

### `rewrite_query`

| | |
|---|---|
| Reads | `user_query`, `anime_title`, `anime_year`, `doc_grades`, `retry_count` |
| Writes | `rewritten_query`, `retry_count` (incremented) |
| Model | gpt-oss-20b, `reasoning_effort="low"`, plain string output |
| LangSmith tag | `{"stage": "rewrite", "llm": "gpt-oss-20b"}` |

The failed docs' one-line reasons are fed back into the prompt so the rewrite targets *why* retrieval failed.

**Failure mode:** empty output → fall back to `f"{user_query} anime"`.

In [ ]:
REWRITE_PROMPT = """The user asked: {query}
We assumed it was about: {title} ({year})

Our retrieval failed. Reasons the graded documents gave:
{reasons}

Rewrite the search query to be more likely to find the right answer.
Drop the title if it seems wrong. Add "anime" if missing.
Return ONLY the rewritten query, no quotes, no explanation."""

def rewrite_query(state: CRAGState, config: RunnableConfig) -> dict:
    cfg = trace_config(
        config, "rewrite_query",
        ["crag", "stage:rewrite", "llm:gpt-oss-20b"],
        user_query=state["user_query"],
    )
    failed = [g for g in state.get("doc_grades", []) if not g["relevant"]]
    reasons = "\n".join(f"- {g['reason']}" for g in failed) or "- nothing relevant was retrieved"
    prompt = REWRITE_PROMPT.format(
        query=state["user_query"],
        title=state["anime_title"],
        year=state.get("anime_year") or "unknown",
        reasons=reasons,
    )
    resp = llm_fast.invoke([("user", prompt)], config=cfg)   # .content only - reasoning ignored
    rewritten = (resp.content or "").strip().strip('"').strip() or f'{state["user_query"]} anime'
    # rewriting consumes the one allowed retry
    return {"rewritten_query": rewritten, "retry_count": state.get("retry_count", 0) + 1}

### `bump_retry`

A three-line node that exists for one reason: **loop termination on the AMBIGUOUS branch.**

- `web_search` never increments `retry_count` itself - folding the increment in would give the same node two behaviors and make traces harder to read.
- `rewrite_query` bumps because it consumes the retry.
- AMBIGUOUS skips rewriting (the query is fine, we just need more evidence), so `bump_retry` bumps instead - without it, `AMBIGUOUS → web_search → grade → AMBIGUOUS → ...` never terminates.

In [ ]:
def bump_retry(state: CRAGState, config: RunnableConfig) -> dict:
    # config accepted for a uniform node signature; no child runnable to forward it to
    cfg = trace_config(config, "bump_retry", ["crag", "stage:bump_retry"], user_query=state["user_query"])
    return {"retry_count": state.get("retry_count", 0) + 1}

### `generate`

| | |
|---|---|
| Reads | `user_query`, `retrieved_docs`, `web_used`, `retrieval_quality`, `retry_count` |
| Writes | `final_answer` |
| Model | openai/gpt-oss-120b, `reasoning_effort="low"`, plain markdown output |
| LangSmith tag | `{"stage": "generate", "llm": "gpt-oss-120b"}` |

Cites documents inline as `[1]`, `[2]`, ... and says plainly when the documents don't contain the answer - no invented facts.

**Prepend rules** (exact strings, no variation - makes tests deterministic):

1. `web_used` → `> Answer used web search fallback.\n\n`
2. `retry_count >= 1` AND `retrieval_quality != "CORRECT"` → `> Low confidence - retrieval was incomplete.\n\n`
3. Both apply → stack them in that order.

**Failure mode:** no docs → hardcoded string, **no LLM call** - saves tokens and makes the failure path testable.

In [ ]:
def generate(state: CRAGState, config: RunnableConfig) -> dict:
    cfg = trace_config(
        config, "generate",
        ["crag", "stage:generate", "llm:gpt-oss-120b"],
        user_query=state["user_query"],
    )
    docs = state.get("retrieved_docs", [])
    if not docs:
        # no LLM call - the failure path stays cheap and testable
        return {"final_answer": "I couldn't find reliable information about this. "
                                "The retrieval pipeline returned nothing relevant."}

    blocks = []
    for i, d in enumerate(docs, start=1):   # citations are 1-based
        m = d.metadata
        blocks.append(
            f'[{i}] (source={m.get("source")}, section={m.get("section")}, url={m.get("url")})\n{d.page_content}'
        )
    prompt = f"""Answer the user's question using ONLY the documents below.
Cite documents inline as [1], [2], ... corresponding to their number.
If the documents don't contain the answer, say so plainly - do not invent facts.

Question: {state["user_query"]}

Documents:
{chr(10).join(blocks)}"""

    resp = llm_smart.invoke([("user", prompt)], config=cfg)   # .content only - reasoning ignored
    answer = resp.content if isinstance(resp.content, str) else str(resp.content)

    warnings = ""
    if state.get("web_used"):
        warnings += "> Answer used web search fallback.\n\n"
    if state.get("retry_count", 0) >= 1 and state.get("retrieval_quality") != "CORRECT":
        warnings += "> Low confidence - retrieval was incomplete.\n\n"
    return {"final_answer": warnings + answer}

## Routing - three conditional edges

### Edge A - after `jikan_lookup`

```python
return "index_and_retrieve" if state["mal_id"] is not None else "web_search"
```

No LLM. One condition. Don't overthink it.

### Edge B - after `grade_documents`

| Quality | Route | Why (from the CRAG paper) |
|---|---|---|
| `CORRECT` | `generate` | answerable now |
| `AMBIGUOUS` | `bump_retry` → `web_search` | some signal exists - **augment** it; the existing query is good, we just need more |
| `INCORRECT` | `rewrite_query` → `web_search` | the query itself is **wrong** - augmenting with the same query would pull the same garbage, so rewrite first |
| `UNKNOWN` | `generate` | fall-through |

**Hard loop guard:** `retry_count >= 1` → `generate`, checked *before* quality. Never retry more than once → at most two Tavily spans per run and `retry_count ∈ {0, 1}`, never 2.

The AMBIGUOUS-vs-INCORRECT distinction matters more than the code itself: wrong routing is the difference between "get more evidence" and "fix the question first."

### Edge C - the loop closes

```text
grade_documents ─┬─ CORRECT ──────────────► generate
                 ├─ AMBIGUOUS ─► bump_retry ─► web_search ─► index_and_retrieve ─► grade_documents
                 └─ INCORRECT ─► rewrite_query (bumps retry) ─► web_search ─► index_and_retrieve ─► grade_documents
```

`web_search` always goes to `index_and_retrieve`, which always goes to `grade_documents` - re-entering Edge B. The `retry_count >= 1` guard ends the loop.

In [ ]:
def route_after_jikan(state: CRAGState) -> Literal["index_and_retrieve", "web_search"]:
    return "index_and_retrieve" if state["mal_id"] is not None else "web_search"

def route_after_grading(state: CRAGState) -> Literal["generate", "bump_retry", "rewrite_query"]:
    # hard loop guard - never retry more than once
    if state["retry_count"] >= 1:
        return "generate"
    q = state["retrieval_quality"]
    if q == "CORRECT":
        return "generate"
    if q == "AMBIGUOUS":
        return "bump_retry"       # augment with the same query; bump so the loop terminates
    if q == "INCORRECT":
        return "rewrite_query"    # the query is wrong - fix it before augmenting
    return "generate"             # UNKNOWN - treat as fall-through

builder = StateGraph(CRAGState)
for node_name, node_fn in {
    "extract_title": extract_title,
    "jikan_lookup": jikan_lookup,
    "web_search": web_search,
    "index_and_retrieve": index_and_retrieve,
    "grade_documents": grade_documents,
    "rewrite_query": rewrite_query,
    "bump_retry": bump_retry,
    "generate": generate,
}.items():
    builder.add_node(node_name, node_fn)

builder.add_edge(START, "extract_title")
builder.add_edge("extract_title", "jikan_lookup")
builder.add_conditional_edges("jikan_lookup", route_after_jikan, ["index_and_retrieve", "web_search"])
builder.add_edge("index_and_retrieve", "grade_documents")
builder.add_conditional_edges(
    "grade_documents", route_after_grading, ["generate", "bump_retry", "rewrite_query"]
)
builder.add_edge("rewrite_query", "web_search")   # rewrite consumes a retry, then augments
builder.add_edge("bump_retry", "web_search")      # AMBIGUOUS bumps, then augments
builder.add_edge("web_search", "index_and_retrieve")
builder.add_edge("generate", END)

graph = builder.compile()
print(graph.get_graph().draw_ascii())   # grandalf renders the graph in-terminal

## Scenarios

Three queries that exercise every branch. Run them, then verify the traces in LangSmith against the checklist below.

In [ ]:
def run_scenario(label: str, query: str) -> dict:
    state = {
        # input
        "user_query": query,
        # extraction
        "anime_title": "", "anime_year": None,
        # mal resolution
        "mal_id": None, "mal_docs": [],
        # corrective path
        "web_docs": [], "web_used": False, "rewritten_query": None, "retry_count": 0,
        # retrieval + grading
        "retrieved_docs": [], "doc_grades": [], "retrieval_quality": "UNKNOWN",
        # output
        "final_answer": "",
    }
    final = graph.invoke(state, config={"run_name": label, "tags": ["crag", f"scenario:{label}"]})
    print("=" * 72)
    print(f"{label}: {query}")
    print(f"  title={final['anime_title']!r} year={final['anime_year']} mal_id={final['mal_id']}")
    print(f"  quality={final['retrieval_quality']} retry_count={final['retry_count']} "
          f"web_used={final['web_used']} rewritten={final['rewritten_query']!r}")
    print(f"  docs: retrieved={len(final['retrieved_docs'])} mal={len(final['mal_docs'])} web={len(final['web_docs'])}")
    print(f"  grades: {[(g['index'], g['relevant']) for g in final['doc_grades']]}")
    print("-" * 72)
    print(final["final_answer"])
    return final

**Scenario 1** - clean MAL path. Expected spans in order: `extract_title` → `jikan_lookup` → `index_and_retrieve` → `grade_documents` → `generate`. Zero Tavily spans, `retry_count == 0`.

If you see a Tavily span here, `jikan_lookup` failed and you have a title-matching bug - check the WRatio threshold and whether `/anime?q=` returns `Steins;Gate` with the punctuation (it does; the match score of "Steins;Gate" vs "Steins;Gate" should be 100).

In [ ]:
# Scenario 1 - clean MAL path, zero web search
s1 = run_scenario("scenario_1", "What is Steins;Gate about?")
assert s1["mal_id"] is not None, "title must resolve to a MAL entry"
assert not s1["web_used"], "clean path must not touch Tavily"
assert s1["retry_count"] == 0, "no retries on the clean path"

**Scenario 2** - no title in the query (`extract_title` returns `""` or a hallucinated guess → `jikan_lookup` misses → `mal_id=None`) → corrective path fires immediately. Two Tavily spans max - three means the loop guard is broken. Final `retry_count` must be 0 or 1, never 2.

In [ ]:
# Scenario 2 - title miss → web search first, loop guard caps retries
s2 = run_scenario("scenario_2", "anime about a microwave time machine and lab members")
assert s2["mal_id"] is None, "no title to match - jikan must miss"
assert s2["web_used"], "corrective path must fire"
assert s2["retry_count"] <= 1, "loop guard must cap retries at 1"

**Scenario 3** - the interesting trace. Expected spans: `extract_title` (title "Cowboy Bebop") → `jikan_lookup` (matches, fetches `/full` and `/characters`) → `index_and_retrieve` (metadata + staff rank highest) → `grade_documents` (metadata relevant, staff relevant - composer is in staff; synopsis maybe not, characters not) → **AMBIGUOUS** → `bump_retry` → `web_search` → `index_and_retrieve` → `grade_documents` → `generate`.

This is AMBIGUOUS routing working correctly: even though the answer came from MAL, the grader marked the synopsis irrelevant to the music question, which pulls the average down, which triggers augmentation. If your grader marks everything relevant all the time, this scenario won't fire - raise its `reasoning_effort` or tighten the prompt.

In [ ]:
# Scenario 3 - AMBIGUOUS after a successful MAL lookup → bump_retry → web → generate
s3 = run_scenario("scenario_3", "What year did Cowboy Bebop air and who composed its music?")
assert s3["mal_id"] is not None, "title must resolve to a MAL entry"
assert s3["retry_count"] <= 1, "loop guard must cap retries at 1"
print("\nretrieval_quality:", s3["retrieval_quality"])

## What to check after the notebook runs - LangSmith

Filter by project `multi-query-rag` or tag `crag`. Four verification steps:

1. **Filter by tag `stage:extract`** - every extract span should have a title in the output. If any is `""` when the user gave a clear title, the prompt is bad.
2. **Filter by tag `stage:grade`** - check that `reason` strings are one sentence and specific. "The document is relevant" is a bad grade; the LLM is phoning it in - tighten the prompt.
3. **Sort by latency** - `generate` should be the slowest (120B), `extract_title` fastest. If `grade_documents` is slowest, the batch is too big or `reasoning_effort` should drop to `low`.
4. **Total tokens per scenario** - Scenario 1 should be ~4k input / ~500 output. If it's 20k+, something is being stuffed that shouldn't be (likely full character lists) - `jikan_lookup` already truncates characters and staff to the top 15 entries each.

| Scenario | Expect in LangSmith |
|---|---|
| 1 - "What is Steins;Gate about?" | extract → jikan → retrieve → grade → generate; **zero** Tavily spans; `retry_count == 0` |
| 2 - "anime about a microwave time machine and lab members" | extract (miss) → jikan (`mal_id=None`) → web → retrieve → grade → (rewrite →) web → retrieve → grade → generate; **≤2 Tavily spans**; `retry_count ∈ {0,1}` |
| 3 - "What year did Cowboy Bebop air and who composed its music?" | MAL path + AMBIGUOUS → `bump_retry` → one Tavily span → generate; shows augmentation after a partial success |